# Notebook 04 — Sequential Transfer Learning (Aerial -> Underwater): the "Mixed" baseline

**Paper artifact:** the *Mixed (Sequential Transfer)* configurations of the model comparison (`tab:model_metrics_summary`) and the paper's **negative-transfer** discussion that motivates the Direct Transfer design.

This notebook trains the three *Mixed* detectors — the Nano, Medium and Large YOLO26 backbones — with a **two-phase Sequential Transfer Learning** protocol: a model is first pre-trained on an out-of-domain **aerial** corrosion dataset (source) and then fine-tuned on the **underwater** target domain (low learning rate, frozen backbone). The resulting `modelo-mixto-{n,m,l}.pt` checkpoints are exactly the ones evaluated in Notebook 05.

**Motivation (negative transfer).** The paper's hypothesis is that aerial corrosion features are distributionally *antagonistic* to the underwater domain, so seeding the optimisation with an aerial expert should *degrade* rather than help underwater performance. This notebook produces that baseline; the collapse is then quantified in the comparison table (the *Mixed* rows score F1 <= 0.28, far below Direct Transfer), turning the hypothesis into an empirical result.

---

| | |
|---|---|
| **Inputs** | `dataset_aereo/` (aerial source) and `dataset_yolo/` (underwater target); pretrained `yolo26{n,m,l}.pt` |
| **Output** | three sequential-transfer checkpoints `modelos_entrenados/modelo-mixto-{n,m,l}.pt` (consumed by Notebook 05) |
| **Environment** | Ultralytics · PyTorch 2.x · CUDA · NVIDIA RTX 4090 |

> **Note on outputs.** All code cells and their outputs are preserved exactly as executed on the original GPU run; the Ultralytics training console logs appear in their original (English) form and are not edited. Re-running under `requirements.txt` reproduces the same checkpoints.

In [1]:
import shutil
from pathlib import Path
import torch
from ultralytics import YOLO
import os

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/home/user/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


## 1 · Global configuration and dataset YAMLs

Set the global hyper-parameters shared by all three models and write the two temporary dataset descriptors. The **aerial** YAML reuses its `train` split as `val` (the source set has no validation partition), while the **underwater** YAML points at the target domain used for fine-tuning and final test-time validation.

In [2]:
# --- GLOBAL CONFIGURATION ---
# Make sure these paths exist
ACUATICO_DIR = Path("./dataset_yolo")       
AEREO_DIR = Path("./dataset_aereo")         
OUTPUT_DIR = Path("./modelos_entrenados")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Global parameters
PROJECT_NAME = "Sequential_Transfer_Benchmark"
EPOCHS_AEREO = 30         
EPOCHS_FINE_TUNE = 50     

print(f"\n{'='*60}")
print(f" STRATEGY: SEQUENTIAL TRANSFER LEARNING (MULTI-MODEL)")
print(f" Phase 1: Learn Aerial Corrosion -> Phase 2: Adapt to Underwater")
print(f"{'='*60}")

# --- YAML PREPARATION ---

# 1. Create temporary Aerial YAML (WITH THE VALIDATION FIX)
yaml_aereo = Path("temp_aereo.yaml")
with open(yaml_aereo, "w") as f:
    # TRICK: we use 'train' also as 'val' because the aerial dataset has no split
    # This avoids the "missing path .../val" error
    train_path = (AEREO_DIR / "images/train").absolute()
    
    f.write(f"path: {AEREO_DIR.absolute()}\n")
    f.write(f"train: {train_path}\n")
    f.write(f"val: {train_path}\n")  # <--- THIS IS THE FIX
    f.write("names:\n  0: corrosion")

# 2. Create Underwater YAML (using the original clean dataset)
yaml_acuatico = Path("temp_acuatico.yaml")
with open(yaml_acuatico, "w") as f:
    # We assume dataset_yolo has the correct structure (images/train, images/val)
    # If K-Fold was used earlier, the original dataset_yolo may not have a fixed 'val'.
    # We will use dataset_yolo_aug/dataset.yml if it exists, otherwise point to the original.
    
    # Option A: use the original dataset directly (YOLO performs an automatic split if only train is present)
    f.write(f"path: {ACUATICO_DIR.absolute()}\n")
    f.write("train: images/train\n")
    
    # Check whether val exists; if not, use train (YOLO will split automatically)
    if (ACUATICO_DIR / "images/val").exists():
        f.write("val: images/val\n")
    elif (ACUATICO_DIR / "images/valid").exists():
        f.write("val: images/valid\n")
    else:
        # If there is no val, use train and let YOLO complain or fall back to train
        f.write("val: images/train\n") 
        
    f.write("names:\n  0: corrosion")

print("✅ YAML configuration files created.")


 STRATEGY: SEQUENTIAL TRANSFER LEARNING (MULTI-MODEL)
 Phase 1: Learn Aerial Corrosion -> Phase 2: Adapt to Underwater
✅ YAML configuration files created.


## 2 · Model 1 of 3 — NANO (yolo26n)

**Batch size:** 64 (the lightweight backbone tolerates a large batch).

Phase 1 pre-trains `yolo26n` on the aerial source domain; Phase 2 fine-tunes those *aerial-expert* weights on the underwater target with a low learning rate (`lr0=0.005`) and a frozen backbone (`freeze=10`). The final checkpoint is exported as `modelo-mixto-n.pt`.

In [3]:
MODEL_ARCH = "yolo26n.pt"
BATCH_SIZE = 64
model_name_clean = "yolo26n"

print(f"\n\n{'#'*60}")
print(f" STARTING PROCESS FOR MODEL: {model_name_clean}")
print(f"{'#'*60}")

# --- PHASE 1: AERIAL PRE-TRAINING ---
print(f"\n>>> [{model_name_clean}] PHASE 1: TRAINING ON AERIAL DOMAIN (SOURCE)")

model_phase1 = YOLO(MODEL_ARCH)
model_phase1.train(
    data=str(yaml_aereo),
    epochs=EPOCHS_AEREO,
    imgsz=640,
    batch=BATCH_SIZE,
    project=PROJECT_NAME,
    name=f"phase1_aerial_{model_name_clean}",
    exist_ok=True,
    verbose=True, 
    optimizer='SGD',
    lr0=0.01
)

# Retrieve the resulting weights from Phase 1
weights_phase1 = Path(PROJECT_NAME) / f"phase1_aerial_{model_name_clean}" / "weights" / "best.pt"
print(f"✅ [{model_name_clean}] Phase 1 complete. 'Aerial-expert' weights saved to: {weights_phase1}")

# --- PHASE 2: UNDERWATER FINE-TUNING ---
print(f"\n>>> [{model_name_clean}] PHASE 2: FINE-TUNING ON UNDERWATER DOMAIN (TARGET)")

if weights_phase1.exists():
    model_phase2 = YOLO(weights_phase1)

    # 3. Train (fine-tuning)
    model_phase2.train(
        data=str(yaml_acuatico),
        epochs=EPOCHS_FINE_TUNE,
        imgsz=640,
        batch=BATCH_SIZE,
        project=PROJECT_NAME,
        name=f"phase2_underwater_{model_name_clean}",
        exist_ok=True,
        optimizer='SGD',
        lr0=0.005,      # Low LR
        lrf=0.1,
        freeze=10,      # Freeze backbone
        augment=True    
    )

    # Save the final result
    final_weights = Path(PROJECT_NAME) / f"phase2_underwater_{model_name_clean}" / "weights" / "best.pt"
    dest = OUTPUT_DIR / f"modelo-mixto-n.pt"
    
    if final_weights.exists():
        shutil.copy(final_weights, dest)
        print(f"\n✅ SEQUENTIAL MIXED MODEL SAVED: {dest}")

        # --- FINAL VALIDATION ---
        print(f"\nValidating the new model {model_name_clean}...")
        test_yaml = './dataset_yolo/dataset.yml' 
        try:
            metrics = model_phase2.val(split='test', data=test_yaml)
            print(f"[{model_name_clean}] Final mAP@50: {metrics.box.map50:.4f}")
        except Exception as e:
            print(f"Could not validate automatically. Error: {e}")
    else:
        print("Error: no weights were generated in Phase 2.")
else:
    print(f"❌ Critical error: the Phase 1 model was not found at {weights_phase1}")



############################################################
 STARTING PROCESS FOR MODEL: yolo26n
############################################################

>>> [yolo26n] PHASE 1: TRAINING ON AERIAL DOMAIN (SOURCE)
Ultralytics 8.4.5 🚀 Python-3.9.13 torch-2.8.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4090, 24215MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=temp_aereo.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mi

## 3 · Model 2 of 3 — MEDIUM (yolo26m)

**Batch size:** 16 (memory / throughput balance).

Same two-phase protocol as the Nano model, exported as `modelo-mixto-m.pt`. This is the Medium-backbone *Mixed* counterpart to the Direct-Transfer *Underwater-Medium* model that wins the comparison — the side-by-side gap is the clearest single illustration of negative transfer in the paper.

In [ ]:
MODEL_ARCH = "yolo26m.pt"
BATCH_SIZE = 16
model_name_clean = "yolo26m"

print(f"\n\n{'#'*60}")
print(f" STARTING PROCESS FOR MODEL: {model_name_clean}")
print(f"{'#'*60}")

# --- PHASE 1: AERIAL PRE-TRAINING ---
print(f"\n>>> [{model_name_clean}] PHASE 1: TRAINING ON AERIAL DOMAIN (SOURCE)")

model_phase1 = YOLO(MODEL_ARCH)
model_phase1.train(
    data=str(yaml_aereo),
    epochs=EPOCHS_AEREO,
    imgsz=640,
    batch=BATCH_SIZE,
    project=PROJECT_NAME,
    name=f"phase1_aerial_{model_name_clean}",
    exist_ok=True,
    verbose=True, 
    optimizer='SGD',
    lr0=0.01
)

# Retrieve the resulting weights from Phase 1
weights_phase1 = Path(PROJECT_NAME) / f"phase1_aerial_{model_name_clean}" / "weights" / "best.pt"
print(f"✅ [{model_name_clean}] Phase 1 complete. 'Aerial-expert' weights saved to: {weights_phase1}")

# --- PHASE 2: UNDERWATER FINE-TUNING ---
print(f"\n>>> [{model_name_clean}] PHASE 2: FINE-TUNING ON UNDERWATER DOMAIN (TARGET)")

if weights_phase1.exists():
    model_phase2 = YOLO(weights_phase1)

    # 3. Train (fine-tuning)
    model_phase2.train(
        data=str(yaml_acuatico),
        epochs=EPOCHS_FINE_TUNE,
        imgsz=640,
        batch=BATCH_SIZE,
        project=PROJECT_NAME,
        name=f"phase2_underwater_{model_name_clean}",
        exist_ok=True,
        optimizer='SGD',
        lr0=0.005,      # Low LR
        lrf=0.1,
        freeze=10,      # Freeze backbone
        augment=True    
    )

    # Save the final result
    final_weights = Path(PROJECT_NAME) / f"phase2_underwater_{model_name_clean}" / "weights" / "best.pt"
    dest = OUTPUT_DIR / f"modelo-mixto-m.pt"
    
    if final_weights.exists():
        shutil.copy(final_weights, dest)
        print(f"\n✅ SEQUENTIAL MIXED MODEL SAVED: {dest}")

        # --- FINAL VALIDATION ---
        print(f"\nValidating the new model {model_name_clean}...")
        test_yaml = './dataset_yolo/dataset.yml' 
        try:
            metrics = model_phase2.val(split='test', data=test_yaml)
            print(f"[{model_name_clean}] Final mAP@50: {metrics.box.map50:.4f}")
        except Exception as e:
            print(f"Could not validate automatically. Error: {e}")
    else:
        print("Error: no weights were generated in Phase 2.")
else:
    print(f"❌ Critical error: the Phase 1 model was not found at {weights_phase1}")



############################################################
 STARTING PROCESS FOR MODEL: yolo26m
############################################################

>>> [yolo26m] PHASE 1: TRAINING ON AERIAL DOMAIN (SOURCE)
New https://pypi.org/project/ultralytics/8.4.6 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.5 🚀 Python-3.9.13 torch-2.8.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4090, 24215MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=temp_aereo.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False

## 4 · Model 3 of 3 — LARGE (yolo26l)

**Batch size:** 8 (the heaviest backbone; the batch is reduced to avoid out-of-memory errors).

Same two-phase protocol, exported as `modelo-mixto-l.pt`. Once all three *Mixed* checkpoints exist in `modelos_entrenados/`, Notebook 05 evaluates them on the held-out underwater test split, where the *Mixed* rows collapse relative to Direct Transfer — the empirical confirmation of the negative-transfer hypothesis.

In [ ]:
MODEL_ARCH = "yolo26l.pt"
BATCH_SIZE = 8
model_name_clean = "yolo26l"

print(f"\n\n{'#'*60}")
print(f" STARTING PROCESS FOR MODEL: {model_name_clean}")
print(f"{'#'*60}")

# --- PHASE 1: AERIAL PRE-TRAINING ---
print(f"\n>>> [{model_name_clean}] PHASE 1: TRAINING ON AERIAL DOMAIN (SOURCE)")

model_phase1 = YOLO(MODEL_ARCH)
model_phase1.train(
    data=str(yaml_aereo),
    epochs=EPOCHS_AEREO,
    imgsz=640,
    batch=BATCH_SIZE,
    project=PROJECT_NAME,
    name=f"phase1_aerial_{model_name_clean}",
    exist_ok=True,
    verbose=True, 
    optimizer='SGD',
    lr0=0.01
)

# Retrieve the resulting weights from Phase 1
weights_phase1 = Path(PROJECT_NAME) / f"phase1_aerial_{model_name_clean}" / "weights" / "best.pt"
print(f"✅ [{model_name_clean}] Phase 1 complete. 'Aerial-expert' weights saved to: {weights_phase1}")

# --- PHASE 2: UNDERWATER FINE-TUNING ---
print(f"\n>>> [{model_name_clean}] PHASE 2: FINE-TUNING ON UNDERWATER DOMAIN (TARGET)")

if weights_phase1.exists():
    model_phase2 = YOLO(weights_phase1)

    # 3. Train (fine-tuning)
    model_phase2.train(
        data=str(yaml_acuatico),
        epochs=EPOCHS_FINE_TUNE,
        imgsz=640,
        batch=BATCH_SIZE,
        project=PROJECT_NAME,
        name=f"phase2_underwater_{model_name_clean}",
        exist_ok=True,
        optimizer='SGD',
        lr0=0.005,      # Low LR
        lrf=0.1,
        freeze=10,      # Freeze backbone
        augment=True    
    )

    # Save the final result
    final_weights = Path(PROJECT_NAME) / f"phase2_underwater_{model_name_clean}" / "weights" / "best.pt"
    dest = OUTPUT_DIR / f"modelo-mixto-l.pt"
    
    if final_weights.exists():
        shutil.copy(final_weights, dest)
        print(f"\n✅ SEQUENTIAL MIXED MODEL SAVED: {dest}")

        # --- FINAL VALIDATION ---
        print(f"\nValidating the new model {model_name_clean}...")
        test_yaml = './dataset_yolo/dataset.yml' 
        try:
            metrics = model_phase2.val(split='test', data=test_yaml)
            print(f"[{model_name_clean}] Final mAP@50: {metrics.box.map50:.4f}")
        except Exception as e:
            print(f"Could not validate automatically. Error: {e}")
    else:
        print("Error: no weights were generated in Phase 2.")
else:
    print(f"❌ Critical error: the Phase 1 model was not found at {weights_phase1}")